# Parsing PDF-MHBs into JSON for general universities in Germany
In order to convert PDFs into JSON and extract all the necessary information out of it, different approaches are viable.
1. Extracting data using NER-models by cutting the PDF, that was converted to MD using LLM, into blocks and then classifying which block means what
2. Extracting data using LLMs, either directly or after OCR with LLM e.g. Docling
3. Extracting data using Agent, either directly or after OCR to MD using LLM e.g. Docling or Mineru
4. Using XML, Regex and NER-models combined with LLMs and agents since most MHBs are very similar as the tool for generating them (flexnow) has a monopoly in this area

In this case option 3 will be used

Installing all needed dependencies:

In [15]:
%%capture
!pip3 install docling
!pip3 install 'smolagents[toolkit]'
!pip3 install tokenizers
!pip3 install pydantic

In [16]:
import os
import sys
os.environ['PYTHONPATH'] = os.path.expanduser('~/mhbai')
sys.path.append(os.path.expanduser('~/mhbai'))

In [17]:
base_path = "/home/gatterle/mhbai/"
page_break = "---Page Break---"
token_input_percentage = 0.3

USER-INPUT:
Please choose the path to a pdf

In [18]:
path_to_pdf = "pdfs/20.pdf"
save_files = True
printing = True

## 1. Converting PDF into .md using docling (mineru is an alternative)

### 1.1 Convert PDF to Markdown

In [ ]:
from docling.document_converter import DocumentConverter

path = base_path + path_to_pdf

# create converter
converter = DocumentConverter()

# convert document to markdown
result = converter.convert(path)
md_doc = result.document.export_to_markdown(page_break_placeholder=page_break)

if save_files:
    with open("test-docling.md", "w") as f:
        f.write(md_doc)
if printing:
    print(md_doc)

In [19]:
# alternatively load previously saved markdown
with open("test-docling.md", "r") as f:
    md_doc = f.read()

### 1.2 Split document into module-chunks

In [20]:
pages = md_doc.split("\n" + page_break + "\n")

### 1.2.3 University of Augsburg specific fine tuning

__this will be done using NER-models like Bert__

Generate Table of Contents (ToC)

In [21]:
toc = [pages[1]]
for index, i in enumerate(pages[2:]):
    if "\n| Modul" in i[:15]:
        break
    toc.append(i)

toc = "\n\n".join(e.strip() for e in toc)
pages = pages[index + 2 :]

Generate modules

In [22]:
# initialize list of modules
modules = []

# group pages into modules
mod = []
for i in pages:
    if "\n| Modul" in i[:15]:
        modules.append(mod)
        mod = [i]
    else:
        mod.append(i)
modules.append(mod)

# combine module pages to single module string and remove empty modules
modules = ["\n\n".join(e.strip() for e in i) for i in modules]
modules = [i for i in modules if i.strip() != ""]

# remove duplicates while preserving order
mods = []
for i in modules:
    if i not in mods:
        mods.append(i)
modules = mods

## 2. Extracting important information using smolagents
1. Creating dataclasses, tools and a prompt
2. Executing smolagents with the previously created md-data

### 2.1.1 Create tools

In [23]:
import json
from ai.information_extraction.data_template import exam_info, time_info, ModuleInfo, ModuleHandbook
if printing:
    print(json.dumps(ModuleHandbook.model_json_schema(), indent=2))

{
  "$defs": {
    "ModuleInfo": {
      "description": "module info class for setting strict type requirements for ai model when extracting information\nabout a module\n\nAttributes:\n    title (str | None): title of the module; often it is found under names like Modulcode\n    module_code (str | None): unique module code; often it is found under names like Titel\n    ects (int | None): number of ECTS credits; often it is found under names like ECTS, LP, CP, Credit Points, Leistungspunkte\n    lecturer (str | None): name of the lecturer; often it is found under names like Dozent, Professor, Modulverantwortlicher, Lehrperson, Dozent, Dozent*in\n    contents (list[str] | None): list of module contents; often it is found under names like Inhalt, Beschreibung\n    goals (list[str] | None): list of learning goals; often it is found under names like Ziele, Lernziele, Kompetenzen\n    requirements (list[str] | None): list of prerequisites; often it is found under names like Anforderungen, Vo

### 2.1.2 Create prompt

In [24]:
from ai.information_extraction.smolagent import prompt

full_prompt = prompt.format(module_handbook=md_doc)

### 2.1.3 Create agent

In [25]:
from smolagents import CodeAgent, LiteLLMModel
from smolagents.default_tools import PythonInterpreterTool, FinalAnswerTool
from ai.information_extraction.tools import ValidateOutput
from ai.information_extraction.prompts import system_prompt, instructions, prompt


# model = LiteLLMModel(model_id="ollama_chat/gemma4:31b")
# model_name = "qwen3.6:35b-a3b"
# model_name = "gemma4:31b"
model_name = "qwen3.5:4b"
context_window = 256000
model = LiteLLMModel(model_id=f"ollama_chat/{model_name}")

# Create an agent with no tools
agent = CodeAgent(
        tools=[FinalAnswerTool(), ValidateOutput()],
        additional_authorized_imports=["json", "pydantic", "typing", "ai.information_extraction.data_template", "ai.information_extraction.tools", "ai.information_extraction.tools.ValidateOutput"],
        instructions=instructions,
        max_steps=5,
        model=model)

agent.prompt_templates["system_prompt"] = agent.prompt_templates["system_prompt"] + "\n\nAdditional rules (German):\n" + system_prompt.format(module_break="--- NEUES MODUL ---")


### 2.1.4 Create inputs
In order to use the size of the context window as best as possible, use the tokenizer from the used ai model for approximating the size a specific input would have in the context window

In [26]:
from tokenizers import Tokenizer

tokenizer = Tokenizer.from_file("gemma-4-31b-tokenizer.json")

mods = [(i, len(tokenizer.encode(i).ids)) for i in modules]
inputs = []
inp = []
inp_size = 0

limit = token_input_percentage * context_window
limit = 5000
for i in mods:
    if inp_size + i[1] < limit:
        inp.append(i[0])
        inp_size += i[1]
    else:
        inputs.append("\n\n--- NEUES MODUL ---\n\n".join(e.strip() for e in inp))
        inp = [i[0]]
        inp_size = i[1]
inputs.append("\n\n--- NEUES MODUL ---\n\n".join(e.strip() for e in inp))

if printing:
    print(f"Number of input chunks: {len(inputs)}")

Number of input chunks: 25


### 2.2 Run agent

In [27]:

model = LiteLLMModel(model_id="ollama_chat/qwen3.5:9b")

In [28]:
results = []
for i in inputs:
    agent = CodeAgent(
            tools=[FinalAnswerTool(), ValidateOutput()],
            additional_authorized_imports=["json", "pydantic", "typing", "ai.information_extraction.data_template", "ai.information_extraction.tools"],
            instructions=instructions,
            max_steps=5,
            model=model)

    agent.prompt_templates["system_prompt"] = agent.prompt_templates["system_prompt"] + "\n\nAdditional rules (German):\n" + system_prompt.format(module_break="--- NEUES MODUL ---")


    # full_prompt = prompt.format(module_handbook=i)
    
    full_prompt = prompt.format(module_handbook=i)
    result = agent.run(full_prompt)
    results.append(result)

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ Modulhandbuch:                                                                                                  │
│ """| Modul MRM-0002: Statistik   | ECTS/LP: 5   |                                                               │
│ |-----------------------------|--------------|                                                                  │
│                                                                                                                 │
│ Version 1.0.0 (seit WS15/16)                                                                                    │
│                                                                                                                 │
│ Modulverantwortliche/r: Prof. Dr. Andreas Rathgeber                                                             │
│                                                                                                                 │
│ ## Inhalte:                                                                                                     │
│                                                                                                                 │
│ Zugehörige Veranstaltung: Stochastik (für WING)                                                                 │
│                                                                                                                 │
│ ## Lernziele/Kompetenzen:                                                                                       │
│                                                                                                                 │
│ Bei vielen wirtschaftswissenschaftlichen Problemstellungen ist die Auswertung von Daten und die                 │
│ Weiterverwendung der Auswertungsergebnisse unerlässlich. Im Rahmen der Veranstaltung sollen die Studierenden    │
│ einerseits die theoretischen Grundlagen sowie die Anwendungsvoraussetzungen der statistischen Verfahren kennen  │
│ lernen und lernen. Anderseits soll auch die Anwendung dieser Verfahren im Mittelpunkt stehen, um den            │
│ Studierenden den Einstieg in das empirische Arbeiten zu erleichtern und sie zur Durchführung eigener            │
│ Datenauswertungen zu befähigen. Hierdurch sind sie auch in der Lage, die gewonnenen Ergebnisse zu               │
│ interpretieren und die Grenzen der verwendeten Methoden zu erkennen.                                            │
│                                                                                                                 │
│ ## Arbeitsaufwand:                                                                                              │
│                                                                                                                 │
│ Gesamt: 150 Std.                                                                                                │
│                                                                                                                 │
│ | Voraussetzungen: Grundkenntnisse aus dem Modul Mathematik für Wirtschaftsingenieure.   | Voraussetzungen:     │
│ Grundkenntnisse aus dem Modul Mathematik für Wirtschaftsingenieure.   | ECTS/LP-Bedingungen: schriftliche       │
│ Prüfung   |                                                                                                     │
│ |----------------------------------------------------------------------------------------|--------------------- │
│ -------------------------------------------------------------------|------------------------------------------- │
│ --|                                                                                                             │
│ | Angebotshäufigkeit: jedes Sommersemester            

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

/home/gatterle/mhbai/.venv/lib/python3.12/site-packages/pydantic/main.py:475: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected 10 fields but got 6: Expected `Message` - serialized value may not be as expected [field_name='message', input_value=Message(content='Thought:...er_specific_fields=None), input_type=Message])
  PydanticSerializationUnexpectedValue(Expected `StreamingChoices` - serialized value may not be as expected [field_name='choices', input_value=Choices(finish_reason='st...r_specific_fields=None)), input_type=Choices])
  return self.__pydantic_serializer__.to_python(


─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  from ai.information_extraction.data_template import ModuleHandbook, ModuleInfo, exam_info, time_info             
  from ai.information_extraction.tools import ValidateOutput                                                       
                                                                                                                   
  # Module 1: MRM-0002: Statistik                                                                                  
  module1 = ModuleInfo(                                                                                            
      title="Statistik",                                                                                           
      module_code="MRM-0002",                                                                                      
      ects=5,                                                                                                      
      lecturer="Prof. Dr. Andreas Rathgeber",                                                                      
      contents=[                                                                                                   
          "Deskriptive Statistik",                                                                                 
          "Einführung",                                                                                            
          "Grundbegriffe der Datenerhebung",                                                                       
          "Auswertungsmethoden für ein- und mehrdimensionales Datenmaterial",                                      
          "Wahrscheinlichkeitsrechnung",                                                                           
          "Kombinatorische Grundlagen",                                                                            
          "Zufallsvorgänge, Ereignisse und Wahrscheinlichkeiten",                                                  
          "Zufallsvariablen, Verteilungen und Verteilungsparameter",                                               
          "Gesetz der großen Zahlen und zentraler Grenzwertsatz",                                                  
          "Induktive Statistik",                                                                                   
          "Grundlagen der induktiven Statistik",                                                                   
          "Punkt-Schätzung",                                                                                       
          "Signifikanztests"                                                                                       
      ],                                                                                                           
      goals=[                                                                                                      
          "Studierende lernen theoretische Grundlagen und Anwendungsvoraussetzungen der statistischen Verfahren",  
          "Anwendung der Verfahren im Mittelpunkt, um Einstieg in empirisches Arbeiten zu erleichtern",            
          "Durchführung eigener Datenauswertungen befähigen",                                                      
          "Ergebnisse interpretieren und Grenzen der Methoden erkennen"                                            
      ],                                                                                                           
      requirements=[                                                                                               
          "Grundkenntnisse aus dem Modul Mathematik für Wirtschaftsingenieure"                                     
      ],                                                                                                           
      expense=[time_info(activity="Gesamt", hours=150)], 

Code parsing failed on line 40 due to: SyntaxError: unterminated string literal (detected at line 40) (<unknown>, 
line 40)
        {"title": "Stochastik (Vorlesung)", "lecturer": "Prof. Dr. Andreas Rathgeber", "ects=ects", "sWS=2},       
^

...

[Step 2: Duration 5.49 seconds| Input tokens: 23,709 | Output tokens: 2,472]

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ Modulhandbuch:                                                                                                  │
│ """| Modul PHM-0191: Technische Physik II   | ECTS/LP: 6   |                                                    │
│ |----------------------------------------|--------------|                                                       │
│                                                                                                                 │
│ Version 1.0.0                                                                                                   │
│                                                                                                                 │
│ Modulverantwortliche/r: Prof. Dr.-Ing. Alois Loidl                                                              │
│                                                                                                                 │
│ ## Lernziele/Kompetenzen:                                                                                       │
│                                                                                                                 │
│ ## Die Studierenden                                                                                             │
│                                                                                                                 │
│ - kennen die grundlegenden Begriffe, Konzepte und Phänomene der Elektrostatik und des Magnetismus; des Weiteren │
│ die Grundbegriffe der Elektrodynamik und der Optik,                                                             │
│ - besitzen Fertigkeiten in der mathematischen Beschreibung elektromagnetischer Phänomene, Modellbildung, der    │
│ Formulierung mathematisch-physikalischer Ansätze und können diese auf Aufgabenstellungen in den genannten       │
│ Bereichen anwenden und                                                                                          │
│ - besitzen Kompetenzen in der selbständigen Bearbeitung von Problemstellungen zu den genannten Themenbereichen. │
│ Sie sind in der Lage, Genauigkeiten von Beobachtung und Analyse einschätzen zu können.                          │
│                                                                                                                 │
│ ## Bemerkung:                                                                                                   │
│                                                                                                                 │
│ Mathematische Hilfsmittel wie Differentiation &amp; Integration, einfache Differentialgleichungen und komplexe  │
│ Zahlen werden je nach Vorkommen in das Modul integriert                                                         │
│                                                                                                                 │
│ ## Arbeitsaufwand:                                                                                              │
│                                                                                                                 │
│ Gesamt: 180 Std.                                                                                                │
│                                                                                                                 │
│ | Voraussetzungen: Die Vorlesung baut auf den Inhalten der Vorlesung Technische Physik I auf.   |               │
│ Voraussetzungen: Die Vorlesung baut auf den Inhalten der Vorlesung Technische Physik I auf.   |                 │
│ ECTS/LP-Bedingungen: schriftliche Prüfung   |                                                                   │
│ |-----------------------------------------------------

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

[Step 1: Duration 42.26 seconds]

KeyboardInterrupt: 

Validate output and convert it to JSON

In [ ]:
res = []
for result in results:
    try:
        result = ModuleHandbook.model_validate(result)
        res.append(result)
    except Exception as e:
        print(f"Validation failed: {e}")

data = [r.model_dump() for r in res]
modules = []
for d in data:
    modules.extend(mods if (mods := d["modules"]) is not None else [])
if printing:
    print(f"Extracted {len(modules)} modules.")
    print(f"{len([i for i in data if i['modules'] is not None])} of {len(results)} extractions were successful.")

In [ ]:
import json

if printing:
    print(json.dumps(data, indent=4, ensure_ascii=False))

## 3. Create confidence-score using Regex and NER-models

In [ ]:
pass

## 4. Save data to database (unia.modules_ai_extracted)

In [ ]:
technique = f"agent-{model_name}"



In [ ]:
print(len(md_doc.split(" ")))